# Python Datetime Fundamentals

Working with dates and times is foundational for time series analysis. Before we can index,
slice, resample, or forecast a time series, we need to represent *when* each observation
occurred.

Python provides several layers of datetime support, each suited to different tasks:

| Layer | Module | Strength |
|-------|--------|----------|
| Standard library | `datetime` | General-purpose, timezone-aware date/time objects |
| NumPy | `numpy.datetime64` | Vectorised arithmetic on large arrays of dates |
| Pandas | `pd.Timestamp` / `DatetimeIndex` | The workhorse for labelling and slicing time series |

This notebook walks through all three, building from the basics to the Pandas tools we will
use throughout the rest of the course.

---
## 1. Python's `datetime` Module

The standard library `datetime` module gives us four core types:

- **`date`** — a calendar date (year, month, day)
- **`time`** — a wall-clock time (hour, minute, second, microsecond)
- **`datetime`** — date + time combined
- **`timedelta`** — a duration (the difference between two dates/datetimes)

In [ ]:
from datetime import date, time, datetime, timedelta

### 1.1 `date` — Calendar Dates

In [ ]:
# Today's date
today = date.today()
print(f"Today: {today}")
print(f"Year:  {today.year}")
print(f"Month: {today.month}")
print(f"Day:   {today.day}")

In [ ]:
# Create a specific date
moon_landing = date(1969, 7, 20)
print(f"Moon landing: {moon_landing}")

# Day of the week: .weekday() returns 0=Mon ... 6=Sun
# .isoweekday() returns 1=Mon ... 7=Sun
print(f"weekday():    {moon_landing.weekday()}  (0=Mon)")
print(f"isoweekday(): {moon_landing.isoweekday()}  (1=Mon)")

### 1.2 `time` — Wall-Clock Time

In [ ]:
# Create a time object
market_open = time(9, 30, 0)  # 09:30:00
print(f"Market opens at: {market_open}")
print(f"Hour:   {market_open.hour}")
print(f"Minute: {market_open.minute}")
print(f"Second: {market_open.second}")

# With microseconds
precise = time(14, 30, 15, 123456)
print(f"\nPrecise time: {precise}")
print(f"Microsecond:  {precise.microsecond}")

### 1.3 `datetime` — Date + Time Combined

In [ ]:
# Current date and time
now = datetime.now()
print(f"Now:   {now}")

# .today() is equivalent to .now() without timezone info
print(f"Today: {datetime.today()}")

In [ ]:
# Create a specific datetime
launch = datetime(2024, 3, 15, 14, 30, 0)
print(f"Launch: {launch}")
print(f"Date part: {launch.date()}")
print(f"Time part: {launch.time()}")

In [ ]:
# Combine a date and time into a datetime
d = date(2024, 12, 25)
t = time(8, 0, 0)
christmas_morning = datetime.combine(d, t)
print(f"Christmas morning: {christmas_morning}")

In [ ]:
# Access all components
print(f"Year:        {launch.year}")
print(f"Month:       {launch.month}")
print(f"Day:         {launch.day}")
print(f"Hour:        {launch.hour}")
print(f"Minute:      {launch.minute}")
print(f"Second:      {launch.second}")
print(f"Microsecond: {launch.microsecond}")

### 1.4 `timedelta` — Date Arithmetic

A `timedelta` represents a duration. You get one by subtracting two dates/datetimes,
or you can create one directly.

In [ ]:
# How many days until the end of the year?
today = date.today()
end_of_year = date(today.year, 12, 31)
remaining = end_of_year - today
print(f"Today:            {today}")
print(f"End of year:      {end_of_year}")
print(f"Days remaining:   {remaining.days}")
print(f"timedelta object: {remaining}")

In [ ]:
# What date is 90 days from today?
future = today + timedelta(days=90)
print(f"90 days from {today} is {future}")

# You can also use weeks
two_weeks_ago = today - timedelta(weeks=2)
print(f"Two weeks ago: {two_weeks_ago}")

In [ ]:
# Timedeltas work with datetimes too
now = datetime.now()
meeting = now + timedelta(hours=3, minutes=30)
print(f"Now:     {now}")
print(f"Meeting: {meeting}")
print(f"In:      {meeting - now}")

---
## 2. Formatting and Parsing with `strftime` / `strptime`

Two complementary methods for converting between datetime objects and strings:

- **`strftime`** ("string **f**ormat time") — datetime **to** string
- **`strptime`** ("string **p**arse time") — string **to** datetime

### Common format codes

| Code | Meaning | Example |
|------|---------|---------|
| `%Y` | 4-digit year | 2024 |
| `%m` | Zero-padded month | 03 |
| `%d` | Zero-padded day | 15 |
| `%H` | Hour (24-hour) | 14 |
| `%M` | Minute | 30 |
| `%S` | Second | 05 |
| `%A` | Full day name | Friday |
| `%B` | Full month name | March |
| `%j` | Day of year | 075 |

In [ ]:
# strftime — format a datetime as a string
dt = datetime(2024, 3, 15, 14, 30, 0)

print(dt.strftime("%Y-%m-%d"))              # ISO format
print(dt.strftime("%B %d, %Y"))             # "March 15, 2024"
print(dt.strftime("%A, %B %d, %Y %H:%M"))  # Full verbose
print(dt.strftime("Day %j of %Y"))          # Day of year

In [ ]:
# strptime — parse a string into a datetime
# The format string must match the input exactly

# "March 15, 2024"
dt1 = datetime.strptime("March 15, 2024", "%B %d, %Y")
print(f"Parsed: {dt1}")

# "15/03/2024" (European day/month/year)
dt2 = datetime.strptime("15/03/2024", "%d/%m/%Y")
print(f"Parsed: {dt2}")

# ISO 8601 with time
dt3 = datetime.strptime("2024-03-15T14:30:00", "%Y-%m-%dT%H:%M:%S")
print(f"Parsed: {dt3}")

# Also available: datetime.fromisoformat() for ISO strings (Python 3.7+)
dt4 = datetime.fromisoformat("2024-03-15T14:30:00")
print(f"ISO:    {dt4}")

---
## 3. Timezone Handling with `zoneinfo` (Python 3.9+)

Real-world time series data often comes from multiple timezones. Financial markets,
IoT sensors, and global operations all require timezone-aware datetimes.

Python 3.9 introduced `zoneinfo`, which replaces the third-party `pytz` library.

In [ ]:
from zoneinfo import ZoneInfo

# Create a timezone-aware datetime
nyc = datetime(2024, 3, 15, 9, 30, tzinfo=ZoneInfo("America/New_York"))
print(f"NYC market open: {nyc}")

# Convert to other timezones
london = nyc.astimezone(ZoneInfo("Europe/London"))
tokyo = nyc.astimezone(ZoneInfo("Asia/Tokyo"))
print(f"London:          {london}")
print(f"Tokyo:           {tokyo}")

In [ ]:
# These all represent the same instant in time
print(f"NYC == London? {nyc == london}")
print(f"NYC == Tokyo?  {nyc == tokyo}")

In [ ]:
# Make an existing naive datetime timezone-aware
naive = datetime(2024, 6, 15, 12, 0, 0)
aware = naive.replace(tzinfo=ZoneInfo("US/Pacific"))
print(f"Naive: {naive}")
print(f"Aware: {aware}")
print(f"As UTC: {aware.astimezone(ZoneInfo('UTC'))}")

> **Best practice:** Always store time series data in UTC internally, and convert to
> local timezones only for display or when business logic requires it.

---
## 4. NumPy `datetime64` and `timedelta64`

NumPy provides its own datetime type optimised for **vectorised operations** on large
arrays of dates. Where the stdlib `datetime` is great for single values, NumPy shines
when you have millions of timestamps.

In [ ]:
import numpy as np

# Create a datetime64
d = np.datetime64("2024-01-15")
print(f"Date: {d}")
print(f"Type: {d.dtype}")

In [ ]:
# Different precisions
print(f"Year:       {np.datetime64('2024', 'Y')}")
print(f"Month:      {np.datetime64('2024-03', 'M')}")
print(f"Day:        {np.datetime64('2024-03-15', 'D')}")
print(f"Hour:       {np.datetime64('2024-03-15T14', 'h')}")
print(f"Nanosecond: {np.datetime64('2024-03-15T14:30:00.000000001', 'ns')}")

In [ ]:
# Arithmetic with timedelta64
start = np.datetime64("2024-01-01")
end = start + np.timedelta64(90, "D")
print(f"Start: {start}")
print(f"90 days later: {end}")

# Difference
diff = np.datetime64("2024-12-31") - np.datetime64("2024-01-01")
print(f"Days in 2024: {diff}")

In [ ]:
# Creating date ranges with np.arange
months = np.arange("2024-01", "2024-07", dtype="datetime64[M]")
print(f"Monthly range:\n{months}")

days = np.arange("2024-03-01", "2024-03-08", dtype="datetime64[D]")
print(f"\nDaily range:\n{days}")

---
## 5. Pandas `Timestamp` and `DatetimeIndex`

Pandas builds on NumPy's `datetime64` to provide the richest datetime support in the
Python ecosystem. `pd.Timestamp` is the scalar type; `pd.DatetimeIndex` is the array
type used to index time series.

In [ ]:
import pandas as pd

# Create a Timestamp
ts = pd.Timestamp("2024-01-15")
print(f"Timestamp: {ts}")
print(f"Type:      {type(ts)}")

In [ ]:
# Rich component access
ts = pd.Timestamp("2024-03-15 14:30:00")
print(f"Year:       {ts.year}")
print(f"Month:      {ts.month}")
print(f"Day:        {ts.day}")
print(f"Hour:       {ts.hour}")
print(f"Day name:   {ts.day_name()}")
print(f"Month name: {ts.month_name()}")
print(f"Quarter:    {ts.quarter}")
print(f"Day of year:{ts.day_of_year}")
print(f"Week:       {ts.isocalendar()[1]}")

### 5.1 `pd.to_datetime()` — Parsing Dates

The Swiss army knife for converting strings, lists, and columns to datetime.

In [ ]:
# Parse a single string (pandas infers the format)
print(pd.to_datetime("2024-03-15"))
print(pd.to_datetime("March 15, 2024"))
print(pd.to_datetime("15/03/2024", dayfirst=True))

In [ ]:
# Parse a list of strings
dates = pd.to_datetime(["2024-01-01", "2024-02-01", "2024-03-01"])
print(dates)
print(f"Type: {type(dates)}")

In [ ]:
# Use format= for explicit parsing (faster and unambiguous)
dates = pd.to_datetime(
    ["15-Mar-2024", "16-Mar-2024", "17-Mar-2024"],
    format="%d-%b-%Y"
)
print(dates)

In [ ]:
# Handle bad data with errors='coerce' — invalid dates become NaT (Not a Time)
messy = ["2024-01-01", "not-a-date", "2024-03-01", ""]
parsed = pd.to_datetime(messy, errors="coerce")
print(parsed)
print(f"\nNaT count: {parsed.isna().sum()}")

### 5.2 `pd.date_range()` — Generating Regular Date Sequences

This is how we create the regular time axes that underpin time series analysis.

#### Modern frequency aliases (pandas 2.2+)

| Old (deprecated) | New | Meaning |
|------------------|-----|----------|
| `'M'` | `'ME'` | Month **E**nd |
| `'Q'` | `'QE'` | Quarter **E**nd |
| `'A'` / `'Y'` | `'YE'` | Year **E**nd |
| `'H'` | `'h'` | Hour |
| `'T'` | `'min'` | Minute |
| `'S'` | `'s'` | Second |
| `'MS'` | `'MS'` | Month **S**tart (unchanged) |
| `'BM'` | `'BME'` | Business Month End |

In [ ]:
# Daily range with start and end
daily = pd.date_range(start="2024-01-01", end="2024-01-10", freq="D")
print("Daily:")
print(daily)

In [ ]:
# Weekly range with periods
weekly = pd.date_range(start="2024-01-01", periods=6, freq="W")
print("Weekly:")
print(weekly)

In [ ]:
# Monthly (month-end) — use 'ME' not the deprecated 'M'
monthly = pd.date_range(start="2024-01-01", periods=6, freq="ME")
print("Month-end:")
print(monthly)

In [ ]:
# Quarterly and yearly
quarterly = pd.date_range(start="2024-01-01", periods=4, freq="QE")
print("Quarterly:")
print(quarterly)

yearly = pd.date_range(start="2020-01-01", periods=5, freq="YE")
print(f"\nYearly:")
print(yearly)

In [ ]:
# Hourly and minute frequencies — use 'h' and 'min' (not 'H' or 'T')
hourly = pd.date_range(start="2024-01-01", periods=6, freq="h")
print("Hourly:")
print(hourly)

every_15_min = pd.date_range(start="2024-01-01 09:00", periods=5, freq="15min")
print(f"\nEvery 15 minutes:")
print(every_15_min)

In [ ]:
# Business days (weekdays only)
business = pd.date_range(start="2024-03-11", periods=7, freq="B")
print("Business days:")
for d in business:
    print(f"  {d.strftime('%Y-%m-%d %A')}")

### 5.3 `DatetimeIndex` — The Foundation of Time Series in Pandas

When a DataFrame or Series has a `DatetimeIndex`, pandas unlocks powerful time-based
slicing, resampling, and analysis.

In [ ]:
# Create a time series with a DatetimeIndex
np.random.seed(42)
idx = pd.date_range("2024-01-01", periods=365, freq="D")
ts = pd.Series(
    np.random.randn(365).cumsum(),  # random walk
    index=idx,
    name="value"
)
print(ts.head(10))
print(f"\nIndex type: {type(ts.index)}")
print(f"Freq:       {ts.index.freq}")

In [ ]:
# Slice by date strings — partial string indexing
# All of January 2024
print("January 2024:")
print(ts["2024-01"].head())
print(f"... ({len(ts['2024-01'])} days)")

In [ ]:
# Slice a range: March through June
spring = ts["2024-03":"2024-06"]
print(f"March-June: {len(spring)} days")
print(spring.head())

In [ ]:
# Access a specific date
print(f"Value on 2024-07-04: {ts['2024-07-04']:.4f}")

### 5.4 `pd.infer_freq()` — Detecting Frequency Automatically

In [ ]:
# infer_freq detects the frequency of a DatetimeIndex
daily_idx = pd.date_range("2024-01-01", periods=30, freq="D")
monthly_idx = pd.date_range("2024-01-01", periods=12, freq="ME")
hourly_idx = pd.date_range("2024-01-01", periods=48, freq="h")

print(f"Daily:   {pd.infer_freq(daily_idx)}")
print(f"Monthly: {pd.infer_freq(monthly_idx)}")
print(f"Hourly:  {pd.infer_freq(hourly_idx)}")

In [ ]:
# Returns None for irregular data
irregular = pd.DatetimeIndex(["2024-01-01", "2024-01-03", "2024-01-07", "2024-01-08"])
print(f"Irregular: {pd.infer_freq(irregular)}")

---
## 6. Pandas `Period` and `PeriodIndex`

A `Timestamp` represents a **point** in time. A `Period` represents a **span** of time.

| Concept | Type | Example |
|---------|------|---------|
| Moment in time | `Timestamp` | 2024-01-15 14:30:00 |
| Span of time | `Period` | January 2024 (the whole month) |

In [ ]:
# A Period represents a span
p = pd.Period("2024-01", freq="M")
print(f"Period:     {p}")
print(f"Start:      {p.start_time}")
print(f"End:        {p.end_time}")
print(f"Frequency:  {p.freq}")

In [ ]:
# Period arithmetic — adding 1 moves to the next period
print(f"Current:  {p}")
print(f"Next:     {p + 1}")
print(f"Previous: {p - 1}")

In [ ]:
# PeriodIndex
pi = pd.period_range("2024-01", periods=6, freq="M")
print(pi)
print(f"\nType: {type(pi)}")

In [ ]:
# Convert between Timestamp and Period
ts_idx = pd.date_range("2024-01-01", periods=6, freq="ME")
period_idx = ts_idx.to_period("M")
print("DatetimeIndex:")
print(ts_idx)
print(f"\nPeriodIndex:")
print(period_idx)

> **When to use Period vs Timestamp:**
> - Use `Timestamp` / `DatetimeIndex` for most time series work (it is the default in pandas)
> - Use `Period` / `PeriodIndex` when you specifically care about a span (e.g., "Q1 2024"
>   revenue, monthly aggregates where the exact timestamp does not matter)

---
## 7. Summary

| Type | Module | Represents | Best for |
|------|--------|------------|----------|
| `datetime` | stdlib | A single date/time | General Python code, timezone conversions |
| `timedelta` | stdlib | A duration | Date arithmetic |
| `datetime64` | NumPy | A date/time at fixed precision | Vectorised operations on large arrays |
| `timedelta64` | NumPy | A duration at fixed precision | Array-level date arithmetic |
| `Timestamp` | Pandas | A single nanosecond-precision datetime | Indexing and slicing time series |
| `DatetimeIndex` | Pandas | An array of Timestamps | Time series index (the standard choice) |
| `Period` | Pandas | A span of time | When you care about a duration, not a moment |
| `PeriodIndex` | Pandas | An array of Periods | Fiscal quarters, monthly aggregates |

**Key takeaways:**
- For time series analysis in pandas, `pd.Timestamp` and `pd.DatetimeIndex` are the primary tools
- Use `pd.to_datetime()` to parse date strings and `pd.date_range()` to generate regular sequences
- Always use modern pandas frequency aliases (`'ME'`, `'QE'`, `'YE'`, `'h'`, `'min'`) to avoid deprecation warnings
- Store timezone-aware data in UTC; convert to local time only for display
- Use `zoneinfo.ZoneInfo` (not `pytz`) for timezone handling in Python 3.9+